# 05 - Model: Cosine Similarity

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
print(tracks.shape)

(89740, 40)


## Feature space

In [2]:
SOUND_FEATURES = [
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "time_signature",
    "duration_minutes", "tempo_log", "energy_danceability", "mood_score",
    "acoustic_energy_balance", "instrumental_acoustic_score",
]

scaler = StandardScaler()
sound_matrix = scaler.fit_transform(tracks[SOUND_FEATURES])
print(sound_matrix.shape)

(89740, 18)


## Recommend

In [3]:
track_id_to_index = dict(zip(tracks["track_id"], tracks["content_index"]))


def find_similar_tracks(track_id: str, top_n: int = 10) -> pd.DataFrame:
    if track_id not in track_id_to_index:
        raise ValueError(f"Unknown track_id: {track_id}")

    query_index = track_id_to_index[track_id]
    query_genre = tracks.loc[query_index, "track_genre"]

    candidate_mask = (tracks["track_genre"] == query_genre) & (tracks["content_index"] != query_index)
    candidate_indices = tracks.index[candidate_mask].to_numpy()

    similarity = cosine_similarity(sound_matrix[[query_index]], sound_matrix[candidate_indices])[0]
    ranked_order = similarity.argsort()[::-1][:top_n]
    ranked_indices = candidate_indices[ranked_order]

    columns = ["track_id", "track_name", "artists", "album_name", "track_genre", "popularity"]
    recommendations = tracks.loc[ranked_indices, columns].copy()
    recommendations["similarity_score"] = similarity[ranked_order]
    return recommendations.reset_index(drop=True)

In [4]:
for _, seed in tracks.sample(3, random_state=11).iterrows():
    print(f"\n{seed['track_name']} - {seed['artists']} [{seed['track_genre']}]")
    display(find_similar_tracks(seed["track_id"], top_n=5))


Caminos de Ayer - Cuco Sánchez;Fernando Zenaido Maldonado [guitar]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,0VXSstDuTzu6dTYg04SCMs,Corazón Apasionado,Cuco Sánchez,La Gran Colección del 60 Aniversario CBS - Cuc...,guitar,26,0.976973
1,7JI6vO0L5woLjDcrCJvMVt,La Mujer Ladina,Cuco Sánchez;Dueto America,El Revolucionario,guitar,26,0.954728
2,0RLHfz9EazZEt7L0TzFa86,La chancla,Cuco Sánchez,"Corridos, Rancheras, Huapangos y más",guitar,24,0.954044
3,6gANdHuTWi35ExxIvINX2p,"Que Si Te Quiero?, Júralo",Cuco Sánchez,La Gran Colección del 60 Aniversario CBS - Cuc...,guitar,24,0.953860
4,3yWa4RnK34hCiK6RkHe6Q1,Corazoncito tirano,Cuco Sánchez,"Corridos, Rancheras, Huapangos y más",guitar,26,0.952511



Silk Route - Original Mix - ONYVAA [detroit-techno]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,4tkzj544tGv4DwYfqflzw8,1993,Omar S,You Want,detroit-techno,8,0.988617
1,3udUTCId2GHVMGTA3vaff5,Printemps - Original Mix,ONYVAA,Printemps EP,detroit-techno,8,0.986411
2,2WInXs9vEVmpsdcKWC2OEA,Clear - Louderbach All This Space Mix,Cybotron;Louderbach,Clear (Remixes),detroit-techno,4,0.978471
3,4VC4ChQaXEDeOpwuEiXPps,Till We Meet Again - Carl Craig Remix,Kevin Saunderson;Inner City,History Elevate Remixed,detroit-techno,7,0.978188
4,6ajAeHfG5uNzQ4tofAMyWA,You've Got Me - Original Mix,Ryan Crosson;Guti,Marimbaby,detroit-techno,6,0.965955



Långa vägar - Håkan Hellström [goth]


,track_id,track_name,artists,album_name,track_genre,popularity,similarity_score
0,6znXdSMy2nOYC2nuFydQdk,The Weather,New Model Army,From Here,goth,21,0.965919
1,2HTm0c5e8xwLu7GMVpPgxJ,Gråsparven när hon sjunger,Håkan Hellström,Det är så jag säger det,goth,27,0.940766
2,3tXIPBA8SGuCRZWlTHD7tf,Caligula,Hästpojken,Caligula,goth,29,0.932594
3,0aXJCSVfxcelUqmFhlUNeN,Hon är en runaway,Håkan Hellström,Hon är en runaway,goth,27,0.928621
4,67f867INYgEaahq90kqeKE,För en lång lång tid,Håkan Hellström,För sent för Edelweiss,goth,41,0.918851


## Save

In [5]:
np.save(MODEL_DIR / "cosine_feature_matrix.npy", sound_matrix)
joblib.dump(scaler, MODEL_DIR / "cosine_scaler.joblib")

cosine_config = {
    "model": "cosine_similarity",
    "use_case": "similar_song",
    "features": SOUND_FEATURES,
    "candidate_pool": "same_genre",
}
with open(MODEL_DIR / "cosine_config.json", "w", encoding="utf-8") as file:
    json.dump(cosine_config, file, indent=2)